In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
import librosa
from sklearn.model_selection import train_test_split
from tqdm import tqdm

######################################################
# 1) Алфавит (0 = blank, 1..44 = символы)
######################################################
alphabet = {
    '<pad>': 0,
    ' ': 1,
    '#': 2,
    '0': 3,
    '1': 4,
    '2': 5,
    '3': 6,
    '4': 7,
    '5': 8,
    '6': 9,
    '7': 10,
    '8': 11,
    '9': 12,
    'А': 13,
    'Б': 14,
    'В': 15,
    'Г': 16,
    'Д': 17,
    'Е': 18,
    'Ж': 19,
    'З': 20,
    'И': 21,
    'Й': 22,
    'К': 23,
    'Л': 24,
    'М': 25,
    'Н': 26,
    'О': 27,
    'П': 28,
    'Р': 29,
    'С': 30,
    'Т': 31,
    'У': 32,
    'Ф': 33,
    'Х': 34,
    'Ц': 35,
    'Ч': 36,
    'Ш': 37,
    'Щ': 38,
    'Ъ': 39,
    'Ы': 40,
    'Ь': 41,
    'Э': 42,
    'Ю': 43,
    'Я': 44
}
num_classes = len(alphabet)  # = 45

######################################################
# 2) encode_label
######################################################
def encode_label(text: str, alpha: dict):
    return [alpha[ch] for ch in text if ch in alpha]

######################################################
# 3) audio_to_melspectrogram
#    (без аугментации, но "более детальный" если хочешь)
######################################################
def audio_to_melspectrogram(y, sr=16000, 
                            n_mels=64, 
                            n_fft=2048,    # можно увеличить для детализации
                            hop_length=256 # короче шаг => больше фреймов
                           ):
    S = librosa.feature.melspectrogram(y=y, sr=sr, n_fft=n_fft, 
                                       hop_length=hop_length, n_mels=n_mels)
    log_S = librosa.power_to_db(S, ref=np.max)
    # нормируем (0..1)
    log_S_norm = (log_S - log_S.min()) / (log_S.max() - log_S.min() + 1e-6)
    return log_S_norm  # [n_mels, time]

######################################################
# 4) Dataset
######################################################
class MorseAudioCTCDataset(Dataset):
    """
    Без аугментаций — только загружаем wav -> mel
    """
    def __init__(self, df, sr=16000, transform=audio_to_melspectrogram):
        self.df = df.reset_index(drop=True)
        self.sr = sr
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.loc[idx]
        audio_path = "morse_dataset/" + row['id']
        y, _ = librosa.load(audio_path, sr=self.sr)

        # превращаем в mel
        mel = self.transform(y, sr=self.sr)

        mel_tensor = torch.tensor(mel, dtype=torch.float)  # [n_mels, time]
        
        # encode label
        text_label = row['message']
        label_indices = encode_label(text_label, alphabet)
        label_tensor = torch.tensor(label_indices, dtype=torch.long)

        time_dim = mel_tensor.shape[1]

        return mel_tensor, label_tensor, time_dim

######################################################
# 5) ctc_collate_fn
######################################################
def ctc_collate_fn(batch, pool_time_factor=4):
    """
    Собираем батч разной длины => паддим по time
    pool_time_factor=4 => т.к. у нас 2 пула => time//4
    """
    mel_list, label_list, time_list = [], [], []
    for (mel, label, tdim) in batch:
        mel_list.append(mel)
        label_list.append(label)
        time_list.append(tdim)

    max_time = max(m.shape[1] for m in mel_list)
    padded_mels = []
    for mel in mel_list:
        diff = max_time - mel.shape[1]
        if diff>0:
            mel = F.pad(mel, (0,diff), value=0.0)
        mel = mel.unsqueeze(0)  # => [1, n_mels, max_time]
        padded_mels.append(mel)

    audio_batch = torch.stack(padded_mels, dim=0)  # [B, 1, n_mels, max_time]
    labels_concat = torch.cat(label_list, dim=0)

    input_lengths = [(t//pool_time_factor) for t in time_list]
    input_lengths = torch.tensor(input_lengths, dtype=torch.long)
    target_lengths = torch.tensor([len(l) for l in label_list], dtype=torch.long)

    return audio_batch, labels_concat, input_lengths, target_lengths

######################################################
# 6) Модель (более сложная, без аугментаций)
######################################################
class DeepCTCModel(nn.Module):
    """
    3 Conv-блока -> 3-layer BiLSTM -> Linear
    Dropout в conv и LSTM, hidden_size=256
    """
    def __init__(self, num_classes=45, in_channels=1, n_mels=64,
                 hidden_size=256, lstm_layers=3, dropout=0.2):
        super(DeepCTCModel, self).__init__()

        # 1) Conv block #1
        self.conv1 = nn.Sequential(
            nn.Conv2d(in_channels, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Dropout2d(dropout),  # dropout в conv
            nn.MaxPool2d((2,2))  # freq/time -> //2
        )
        # 2) Conv block #2
        self.conv2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Dropout2d(dropout),
            nn.MaxPool2d((2,2))  # freq/time -> //2 (итого //4)
        )
        # 3) Conv block #3 (можно убрать, если GPU слабый)
        self.conv3 = nn.Sequential(
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Dropout2d(dropout)
            # здесь можно добавить MaxPool2d(...) ещё раз,
            # но тогда time сократится до //8
            # для примера оставим без pool
        )

        # freq после 2 пулов => n_mels//4
        # channels => 64 (после conv2)
        # conv3 не делает pool => freq/time не меняется
        # => LSTM вход=64*(n_mels//4)

        self.lstm = nn.LSTM(
            input_size=64*(n_mels//4),
            hidden_size=hidden_size,
            num_layers=lstm_layers,
            dropout=dropout,  # dropout между слоями
            bidirectional=True
        )
        self.fc = nn.Linear(hidden_size*2, num_classes)

    def forward(self, x):
        """
        x: [B, 1, n_mels, time]
        Выход: [time_out, B, num_classes]
        """
        x = self.conv1(x)   # => [B, 32, n_mels//2, time//2]
        x = self.conv2(x)   # => [B, 64, n_mels//4, time//4]
        x = self.conv3(x)   # => [B, 64, n_mels//4, time//4] (no pool here)

        b,c,f,t = x.shape
        # склеим c*f в одну ось
        x = x.view(b, c*f, t)  # => [B, 64*(n_mels//4), time//4]
        x = x.permute(2,0,1)   # => [time//4, B, 64*(n_mels//4)]

        # LSTM
        lstm_out, _ = self.lstm(x)  # => [time//4, B, hidden_size*2]
        logits = self.fc(lstm_out)  # => [time//4, B, num_classes]

        return logits

######################################################
# 7) Тренировочный цикл (с Scheduler)
######################################################
def train_ctc_loop(model, train_loader, val_loader, num_epochs=15, lr=1e-4):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    ctc_loss = nn.CTCLoss(blank=0, reduction='mean', zero_infinity=True)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    # Scheduler: уменьшаем LR в 2 раза, если 3 эпохи нет улучшений в val_loss
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=3, verbose=True
    )

    for epoch in range(num_epochs):
        print(f"\nEpoch {epoch+1}/{num_epochs} (lr={optimizer.param_groups[0]['lr']})")
        model.train()
        train_loss_sum = 0.0
        train_bar = tqdm(train_loader, desc="Train", leave=False)

        for audio_batch, labels_concat, input_lengths, target_lengths in train_bar:
            audio_batch = audio_batch.to(device)
            labels_concat = labels_concat.to(device)
            input_lengths = input_lengths.to(device)
            target_lengths = target_lengths.to(device)

            optimizer.zero_grad()
            logits = model(audio_batch)  # [T, B, C]
            log_probs = F.log_softmax(logits, dim=2)

            loss = ctc_loss(log_probs, labels_concat, input_lengths, target_lengths)
            loss.backward()
            optimizer.step()

            train_loss_sum += loss.item()
            train_bar.set_postfix(loss=loss.item())

        avg_train_loss = train_loss_sum / len(train_loader)

        # Validation
        model.eval()
        val_loss_sum = 0.0
        val_bar = tqdm(val_loader, desc="Val", leave=False)
        with torch.no_grad():
            for audio_batch, labels_concat, input_lengths, target_lengths in val_bar:
                audio_batch = audio_batch.to(device)
                labels_concat = labels_concat.to(device)
                input_lengths = input_lengths.to(device)
                target_lengths = target_lengths.to(device)

                logits = model(audio_batch)
                log_probs = F.log_softmax(logits, dim=2)
                loss = ctc_loss(log_probs, labels_concat, input_lengths, target_lengths)
                val_loss_sum += loss.item()
                val_bar.set_postfix(loss=loss.item())

        avg_val_loss = val_loss_sum / len(val_loader)
        print(f"Epoch {epoch+1}/{num_epochs} | "
              f"Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}")

        # Шаг scheduler (ReduceLROnPlateau)
        scheduler.step(avg_val_loss)

######################################################
# 8) Пример main
######################################################
if __name__ == "__main__":
    df = pd.read_csv("train.csv")

    train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)

    # Dataset (без аугментации)
    train_dataset = MorseAudioCTCDataset(train_df, sr=16000, transform=audio_to_melspectrogram)
    val_dataset   = MorseAudioCTCDataset(val_df,   sr=16000, transform=audio_to_melspectrogram)

    # DataLoaders
    train_loader = DataLoader(
        train_dataset, 
        batch_size=4, 
        shuffle=True, 
        num_workers=0,
        collate_fn=lambda b: ctc_collate_fn(b, pool_time_factor=4)
    )
    val_loader = DataLoader(
        val_dataset, 
        batch_size=4,
        shuffle=False, 
        num_workers=0,
        collate_fn=lambda b: ctc_collate_fn(b, pool_time_factor=4)
    )

    # Модель: 3 conv + 3-lstm
    model = DeepCTCModel(
        num_classes=45,
        in_channels=1,
        n_mels=64,
        hidden_size=256,   # большой
        lstm_layers=3,     # deep
        dropout=0.2
    )

    # Быстрая проверка
    audio_batch, labels_concat, input_lengths, target_lengths = next(iter(train_loader))
    print("audio_batch:", audio_batch.shape)
    print("labels_concat:", labels_concat.shape)
    print("input_lengths:", input_lengths)
    print("target_lengths:", target_lengths)

    # Запускаем обучение
    train_ctc_loop(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        num_epochs=20,
        lr=1e-4    # чуть меньший LR, чтобы избежать переобучения
    )


C:\Users\Admin\Desktop\Код\Питон\morse\.venv\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


audio_batch: torch.Size([4, 1, 64, 501])
labels_concat: torch.Size([39])
input_lengths: tensor([125, 125, 125, 125])
target_lengths: tensor([ 7, 11,  9, 12])

Epoch 1/20 (lr=0.0001)


KeyboardInterrupt: 

In [ ]:
torch.save(model.state_dict(), "Bigger_two.pth")